In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

print("Loading IMDb dataset...")
df = pd.read_csv('IMDB_Dataset.csv')
# print(df.head())

df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(df.head())

# Split data into training and validation sets (e.g., 80% train, 20% val)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# Split the remaining 80% into Train (80%) and Validation (20%) tracking sets.
X_train, X_val, Y_train, Y_val = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42
)
print("*" * 100)
print(f"Dataset loaded. Train samples: {len(train_texts)}, Test samples: {len(test_texts)}")
print(f"Dataset loaded. X_train samples: {len(X_train)}, X_val samples: {len(X_val)}")


Loading IMDb dataset...
                                              review sentiment  label
0  One of the other reviewers has mentioned that ...  positive      1
1  A wonderful little production. <br /><br />The...  positive      1
2  I thought this was a wonderful way to spend ti...  positive      1
3  Basically there's a family where a little boy ...  negative      0
4  Petter Mattei's "Love in the Time of Money" is...  positive      1
****************************************************************************************************
Dataset loaded. Train samples: 40000, Test samples: 10000
Dataset loaded. X_train samples: 32000, X_val samples: 8000


In [2]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# =====================================================================
# INITIALIZE BERT TOKENIZER
# =====================================================================
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Initializing Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_and_tokenize(texts, labels, max_length=256):
    """
    Tokenizes the movie reviews and converts them into BERT input features.

    Parameters:
    - texts: List of raw text movie reviews.
    - labels: List of corresponding integer labels (0 or 1).
    - max_length: Maximum sequence length (BERT supports up to 512 tokens).

    Returns:
    - A PyTorch TensorDataset containing input_ids, attention_masks, and labels.
    """
    cleaned_text = [text.replace("<br/>", " ") for text in texts]

    # Tokenize and convert texts to BERT features
    encodings = tokenizer(cleaned_text, max_length=max_length, add_special_tokens=True, padding="max_length",
                          truncation=True, return_tensors="pt", return_attention_mask=True)

    # print("Tokenizing Movie Review...")
    # print(encodings)

    # Extract structural tensors
    input_ids = encodings['input_ids']
    attention_mask = encodings['attention_mask']
    labels_tensor = torch.tensor(labels)

    # Bundle features together into a structured Dataset format
    return TensorDataset(input_ids, attention_mask, labels_tensor)


# =====================================================================
# CONVERT TO FEATURE DATASETS
# =====================================================================
print("Processing and tokenizing training data (this may take a moment)...")
# Adjust max_length depending on your hardware limits (e.g., 128, 256, or 512)
MAX_LEN = 256

train_dataset = preprocess_and_tokenize(train_texts, train_labels, max_length=MAX_LEN)
val_dataset = preprocess_and_tokenize(X_val, Y_val, max_length=MAX_LEN)
test_dataset = preprocess_and_tokenize(test_texts, test_labels, max_length=MAX_LEN)

# Create iterable data configurations
BATCH_SIZE = 16
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Preprocessing complete!")

# =====================================================================
# VERIFY STRUCUTRE (DEBUG EXAMPLES)
# =====================================================================
# Fetch a single processed item out of the training dataset to inspect
sample_input_ids, sample_mask, sample_label = train_dataset[0]

print("\n--- Verification Sample ---")
print("Input Tensors Shape :", sample_input_ids.shape)
print("Attention Mask Shape:", sample_mask.shape)
print("Label ID            :", sample_label.item())
print("Decoded First 15 Tokens:", tokenizer.decode(sample_input_ids[:15]))

Initializing Tokenizer for sentence-transformers/all-MiniLM-L6-v2...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Processing and tokenizing training data (this may take a moment)...
Preprocessing complete!

--- Verification Sample ---
Input Tensors Shape : torch.Size([256])
Attention Mask Shape: torch.Size([256])
Label ID            : 0
Decoded First 15 Tokens: [CLS] that ' s what i kept asking myself during the many fights, screaming


In [3]:
import torch
import torch.nn as nn
from torch.optim import Adam


# =====================================================================
# SETUP HARDWARE ACCELERATION
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for execution: {device}")

# =====================================================================
# INITIALIZE MODEL & OPTIMIZER CONFIGURATIONS
# =====================================================================
print(f"Loading pre-trained model architecture from {MODEL_NAME}...")
miniLm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
miniLm_model.to(device)
print("*" * 100)

EPOCHS = 3
LEARNING_RATE = 2e-5
optimizer = Adam(miniLm_model.parameters(), lr=LEARNING_RATE, eps=1e-5)
print(optimizer)

total_steps = len(train_dataloader) * EPOCHS
print(total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                            num_training_steps=total_steps)
print(scheduler)
loss_fn = nn.CrossEntropyLoss()


Using device for execution: cpu
Loading pre-trained model architecture from sentence-transformers/all-MiniLM-L6-v2...


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


****************************************************************************************************
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-05
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0
)
7500


In [4]:
# =====================================================================
# 5. MODEL TRAINING LOOP (WITH VALIDATION TRACKING)
# =====================================================================
print("\n Starting Model Training Phase...")

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch + 1} of {EPOCHS} ---")

    # ------------------ TRAINING STEP ------------------
    miniLm_model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_dataloader):
        b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

        optimizer.zero_grad()
        outputs = miniLm_model(input_ids=b_input_ids, attention_mask=b_attention_mask)

        loss = loss_fn(outputs.logits, b_labels)
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(miniLm_model.parameters(), max_norm=1.0)  # Gradient clipping

        optimizer.step()
        scheduler.step()

        if step % 200 == 0 and step > 0:
            print(f"  Batch {step}/{len(train_dataloader)} | Running Loss: {total_train_loss / step:.4f}")

    print(f"▶️ [Train End] Average Epoch Loss: {total_train_loss / len(train_dataloader):.4f}")

    # ------------------ VALIDATION STEP ------------------
    miniLm_model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batch in val_dataloader:
            b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]
            outputs = miniLm_model(input_ids=b_input_ids, attention_mask=b_attention_mask)

            loss = loss_fn(outputs.logits, b_labels)
            total_val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            correct_val += torch.sum(preds == b_labels).item()
            total_val += b_labels.size(0)

    print(
        f"⏸️ [Val End]   Validation Loss: {total_val_loss / len(val_dataloader):.4f} | Accuracy: {(correct_val / total_val) * 100:.2f}%")

print("\n🎉 Model training successfully completed!")


 Starting Model Training Phase...

--- Epoch 1 of 3 ---
  Batch 200/2500 | Running Loss: 0.6958


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =====================================================================
# 6. COMPREHENSIVE TEST EVALUATION METRICS
# =====================================================================
print("\n📊 Commencing Evaluation on Final Unseen Test Set...")
miniLm_model.eval()

all_predictions = []
all_true_labels = []

# Gather raw predictions over the holdout test loop
with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

        outputs = miniLm_model(input_ids=b_input_ids, attention_mask=b_attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        # Move evaluations back to CPU space for scikit-learn metrics
        all_predictions.extend(preds.cpu().numpy())
        all_true_labels.extend(b_labels.cpu().numpy())

# Calculate strict operational performance indicators
test_accuracy = accuracy_score(all_true_labels, all_predictions)
precision, recall, f1_score, _ = precision_recall_fscore_support(
    all_true_labels, all_predictions, average='binary'
)

# Display final processing metrics dashboard
print("\n================ FINAL TEST PERFORMANCE ================")
print(f"  Accuracy  : {test_accuracy * 100:.2f}%")
print(f"  Precision : {precision * 100:.2f}%")
print(f"  Recall    : {recall * 100:.2f}%")
print(f"  F1-Score  : {f1_score * 100:.2f}%")
print("========================================================")

# Save finalized MiniLM deployment architecture
output_dir = "./minilm_imdb_final"
miniLm_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model serialized safely to '{output_dir}'. Ready for inference.")

In [ ]:
# =====================================================================
# 7. PREDICTIONS ON SAMPLE REVIEWS
# =====================================================================
print("\n🔮 Predicting sentiments for sample movie reviews...")

sample_reviews = [
    "The cinematography was brilliant, and the acting was top-notch. Highly recommend!",
    "An absolute waste of time. The plot made no sense and the pacing was horrible.",
    "It wasn't the best movie I've seen, but it had some entertaining moments.",
]

miniLm_model.eval()
label_map = {0: "Negative", 1: "Positive"}

for review in sample_reviews:
    # Preprocess and tokenize the single review
    inputs = tokenizer(
        review,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )

    # Send variables to device
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Run Inference
    with torch.no_grad():
        outputs = miniLm_model(input_ids=input_ids, attention_mask=attention_mask)
        prediction_id = torch.argmax(outputs.logits, dim=1).item()

    print(f"\nReview: \"{review}\"")
    print(f"Predicted Sentiment: {label_map[prediction_id]}")